# MercadoTico 360 · Simulación y carga de datos (CouchDB)

Genera y carga los datos mínimos del Caso 2: **25 000 productos** (8 categorías con atributos variables), **10 000 clientes** y **50 000 pedidos** con varias líneas de detalle.

## Diseño de documentos (Requisito 1: qué se embebe y qué se referencia)

| Documento | `_id` | Estrategia | Justificación |
|---|---|---|---|
| `producto` | `PROD-000001` | Documento propio, `atributos` embebido | Cada categoría tiene atributos distintos; viven dentro del producto sin esquema fijo (Req. 2). |
| `cliente` | `CLI-000001` | Documento propio | Un cliente tiene muchos pedidos; embeberlos haría crecer el documento sin límite. |
| `pedido` | `PED-000001` | `lineas_detalle` **embebidas** + `cliente_id` y `producto_id` **referenciados** | Un pedido se lee siempre completo. Cada línea guarda un *snapshot* (nombre, categoría, precio) para reconstruir la compra aunque el producto cambie. |

Los datos son **reproducibles** (semilla fija) y la carga es **idempotente**: usar el `_id` propio evita duplicados si se corre dos veces.

In [ ]:
import os
import random
import sys
from datetime import date, timedelta
from getpass import getpass

import requests

try:                                   # Opcional: lee un archivo .env si existe
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ----------------------------------------------------------------------
# CONFIGURACIÓN
# Prioridad: variables de entorno (.env) -> si no existen, se pregunta.
# Nunca guarde credenciales en el código ni en el repositorio.
# Ejemplo de .env.example:
#   COUCHDB_HOST=localhost
#   COUCHDB_PORT=5984
#   COUCHDB_USER=admin
#   COUCHDB_PASS=
#   DATABASE_NAME=mercadotico360
# ----------------------------------------------------------------------
def leer_config(nombre, pregunta, defecto=None, secreto=False):
    valor = os.getenv(nombre)
    if valor:
        return valor.strip()
    if secreto:
        return getpass(f"{pregunta}: ").strip()
    sufijo = f" [{defecto}]" if defecto else ""
    return input(f"{pregunta}{sufijo}: ").strip() or defecto

COUCHDB_HOST  = leer_config("COUCHDB_HOST", "Host/IP de CouchDB", "localhost")
COUCHDB_PORT  = leer_config("COUCHDB_PORT", "Puerto de CouchDB", "5984")
COUCHDB_USER  = leer_config("COUCHDB_USER", "Usuario de CouchDB", "admin")
COUCHDB_PASS  = leer_config("COUCHDB_PASS", "Contraseña de CouchDB", secreto=True)
DATABASE_NAME = leer_config("DATABASE_NAME", "Nombre de la base de datos", "mercadotico360")

BASE_URL = f"http://{COUCHDB_HOST}:{COUCHDB_PORT}"
DB_URL   = f"{BASE_URL}/{DATABASE_NAME}"
BULK_URL = f"{DB_URL}/_bulk_docs"

SESSION = requests.Session()           # reutiliza la conexión (más rápido)
SESSION.auth = (COUCHDB_USER, COUCHDB_PASS)

# ----------------------------------------------------------------------
# PARÁMETROS DE LA SIMULACIÓN (mínimos exigidos por el Caso 2)
# ----------------------------------------------------------------------
SEED           = 42        # misma semilla = mismos datos
NUM_PRODUCTOS  = 25_000
NUM_CLIENTES   = 10_000
NUM_PEDIDOS    = 50_000
MIN_CATEGORIAS = 8
BATCH_SIZE     = 5_000

rng = random.Random(SEED)

## Funciones de conexión y carga

In [ ]:
def verificar_o_crear_bd():
    """Comprueba la conexión con CouchDB y crea la base de datos si no existe."""
    try:
        res = SESSION.get(DB_URL, timeout=10)
        if res.status_code == 200:
            print(f"[OK] La base de datos '{DATABASE_NAME}' ya existe.")
        elif res.status_code == 404:
            creada = SESSION.put(DB_URL, timeout=10)
            if creada.status_code not in (201, 202):
                raise RuntimeError(f"No se pudo crear la BD: {creada.status_code} - {creada.text}")
            print(f"[OK] Base de datos '{DATABASE_NAME}' creada.")
        elif res.status_code == 401:
            raise RuntimeError("Usuario o contraseña incorrectos.")
        else:
            raise RuntimeError(f"Error al verificar la BD: {res.status_code} - {res.text}")
    except requests.exceptions.ConnectionError:
        raise RuntimeError(f"No hay conexión con CouchDB en {BASE_URL}. ¿Está el servicio encendido?")


def enviar_lote(documentos):
    """Envía documentos a /_bulk_docs. Devuelve (insertados, ya_existian, errores)."""
    res = SESSION.post(BULK_URL, json={"docs": documentos}, timeout=120)
    res.raise_for_status()
    # _bulk_docs responde 201 aunque falle un documento: hay que revisar cada resultado
    fallos = [r for r in res.json() if "error" in r]
    ya_existian = sum(1 for r in fallos if r["error"] == "conflict")
    return len(documentos) - len(fallos), ya_existian, len(fallos) - ya_existian


def cargar_documentos(documentos):
    """Carga todos los documentos por lotes y muestra el progreso."""
    total = len(documentos)
    insertados = existentes = errores = 0
    for inicio in range(0, total, BATCH_SIZE):
        lote = documentos[inicio:inicio + BATCH_SIZE]
        ok, dup, err = enviar_lote(lote)
        insertados, existentes, errores = insertados + ok, existentes + dup, errores + err
        print(f"  {min(inicio + BATCH_SIZE, total):>6}/{total} procesados")
    print(f"[RESUMEN] nuevos: {insertados} | ya existían: {existentes} | con error: {errores}")

## Productos con atributos heterogéneos (Requisito 2)
Cada categoría define **su propio conjunto de atributos** en un solo lugar (`CATALOGO`). Para agregar una categoría nueva basta con añadir una entrada.

In [ ]:
CATALOGO = {
    "Alimentos": {
        "ejemplo": "Café",
        "atributos": lambda: {
            "peso_g": rng.choice([250, 500, 1000, 2000]),
            "tipo": rng.choice(["Café", "Cereal", "Snack", "Bebida", "Conserva"]),
            "origen": rng.choice(["Costa Rica", "Colombia", "México", "Brasil", "Perú"]),
        },
    },
    "Ropa": {
        "ejemplo": "Camiseta",
        "atributos": lambda: {
            "talla": rng.choice(["XS", "S", "M", "L", "XL"]),
            "color": rng.choice(["Rojo", "Azul", "Negro", "Blanco"]),
            "material": rng.choice(["Algodón", "Poliéster", "Cuero", "Lino"]),
        },
    },
    "Tecnologia": {
        "ejemplo": "Laptop",
        "atributos": lambda: {
            "marca": f"TechBrand-{rng.randint(1, 50)}",
            "ram_gb": rng.choice([4, 8, 16, 32, 64]),            # usado por idx_technology_ram
            "almacenamiento_gb": rng.choice([128, 256, 512, 1024, 2048]),
        },
    },
    "Hogar": {
        "ejemplo": "Lámpara",
        "atributos": lambda: {
            "material": rng.choice(["Metal", "Madera", "Vidrio", "Plástico"]),
            "color": rng.choice(["Blanco", "Negro", "Dorado", "Plateado"]),
            "altura_cm": rng.choice([20, 35, 50, 80, 120, 160]),
        },
    },
    "Cuidado_Personal": {
        "ejemplo": "Shampoo",
        "atributos": lambda: {
            "marca": f"CareBrand-{rng.randint(1, 50)}",
            "volumen_ml": rng.choice([100, 250, 400, 750, 1000]),
            "tipo_cabello": rng.choice(["Graso", "Seco", "Mixto", "Normal", "Rizado"]),
        },
    },
    "Libros": {
        "ejemplo": "Novela",
        "atributos": lambda: {
            "autor": f"Autor {rng.randint(1, 500)}",
            "editorial": rng.choice(["Planeta", "Alfaguara", "Anagrama", "Santillana", "EUNED"]),
            "idioma": rng.choice(["Español", "Inglés", "Francés", "Portugués"]),
        },
    },
    "Mascotas": {
        "ejemplo": "Alimento para mascota",
        "atributos": lambda: {
            "especie": rng.choice(["Perro", "Gato", "Ave", "Pez", "Roedor"]),
            "peso_kg": rng.choice([1, 2, 5, 10, 15, 20]),
            "sabor": rng.choice(["Pollo", "Res", "Pescado", "Cordero", "Mixto"]),
        },
    },
    "Artesanias": {
        "ejemplo": "Taza de cerámica",
        "atributos": lambda: {
            "material": rng.choice(["Cerámica", "Madera", "Metal", "Textil", "Vidrio"]),
            "tecnica": rng.choice(["Torno", "Tallado", "Tejido", "Pintado a mano", "Soplado"]),
            "region_origen": rng.choice(["Guanacaste", "Sarchí", "Guaitil", "Chorotega", "San Vicente de Nicoya"]),
        },
    },
}


def generar_productos(cantidad=NUM_PRODUCTOS):
    """Genera productos repartidos entre todas las categorías del CATALOGO."""
    categorias = list(CATALOGO)
    productos = []
    for i in range(1, cantidad + 1):
        cat = rng.choice(categorias)
        producto_id = f"PROD-{i:06d}"
        productos.append({
            "_id": producto_id,                       # id propio => carga idempotente
            "tipo_doc": "producto",
            "producto_id": producto_id,
            "nombre": f"{CATALOGO[cat]['ejemplo']} {i}",
            "categoria": cat,
            "precio": round(rng.uniform(5.0, 1500.0), 2),
            "stock": rng.randint(0, 500),
            "atributos": CATALOGO[cat]["atributos"](),
        })
    return productos

## Clientes

In [ ]:
PROVINCIAS = ["San José", "Alajuela", "Cartago", "Heredia", "Guanacaste", "Puntarenas", "Limón"]


def generar_clientes(cantidad=NUM_CLIENTES):
    """Genera clientes sintéticos con provincias de Costa Rica."""
    inicio = date(2025, 1, 1)
    clientes = []
    for i in range(1, cantidad + 1):
        cliente_id = f"CLI-{i:06d}"
        clientes.append({
            "_id": cliente_id,
            "tipo_doc": "cliente",
            "cliente_id": cliente_id,
            "nombre": f"Cliente {i}",
            "provincia": rng.choice(PROVINCIAS),
            "email": f"usuario{i}@ejemplo.cr",
            "fecha_registro": (inicio + timedelta(days=rng.randint(0, 364))).isoformat(),
        })
    return clientes

## Pedidos con líneas de detalle anidadas
Cada línea **referencia** el producto (`producto_id`) y además guarda un **snapshot** de nombre, categoría y precio. Así el pedido se puede reconstruir aunque el producto cambie, y las agregaciones por categoría no necesitan consultar el catálogo.

In [ ]:
ESTADOS = ["completado", "pendiente", "cancelado"]


def generar_pedidos(productos, cantidad=NUM_PEDIDOS, num_clientes=NUM_CLIENTES):
    """Genera pedidos de 1 a 4 líneas, tomando los datos de productos reales."""
    inicio = date(2026, 1, 1)
    pedidos = []
    for i in range(1, cantidad + 1):
        lineas = []
        for _ in range(rng.randint(1, 4)):
            prod = rng.choice(productos)
            unidades = rng.randint(1, 3)
            lineas.append({
                "producto_id": prod["producto_id"],          # referencia
                "nombre_snapshot": prod["nombre"],           # snapshot histórico
                "categoria_snapshot": prod["categoria"],
                "cantidad": unidades,
                "precio_unitario": prod["precio"],
                "subtotal": round(prod["precio"] * unidades, 2),
            })

        pedido_id = f"PED-{i:06d}"
        pedidos.append({
            "_id": pedido_id,
            "tipo_doc": "pedido",
            "pedido_id": pedido_id,
            "cliente_id": f"CLI-{rng.randint(1, num_clientes):06d}",   # referencia
            "fecha_pedido": (inicio + timedelta(days=rng.randint(0, 270))).isoformat(),
            "lineas_detalle": lineas,                                  # embebido
            "monto_total": round(sum(l["subtotal"] for l in lineas), 2),
            "estado": rng.choice(ESTADOS),
        })
    return pedidos

## Ejecución
Pasos: validar mínimos → generar → cargar → verificar → mostrar documentos reales.

In [ ]:
print("=== SIMULACIÓN Y CARGA DE DATOS · MERCADOTICO 360 ===\n")

# 0. Validar que se cumplen los mínimos del caso
assert len(CATALOGO) >= MIN_CATEGORIAS, "Se requieren al menos 8 categorías"
assert NUM_PRODUCTOS >= 25_000 and NUM_CLIENTES >= 10_000 and NUM_PEDIDOS >= 50_000, "Volumen por debajo del mínimo"

# 1. Base de datos
verificar_o_crear_bd()

# 2. Generación (la semilla se reinicia para que cada corrida produzca lo mismo)
rng.seed(SEED)
print("\nGenerando datos...")
productos = generar_productos()
clientes  = generar_clientes()
pedidos   = generar_pedidos(productos)
print(f"  productos: {len(productos)} | clientes: {len(clientes)} | pedidos: {len(pedidos)}")

# 3. Carga por lotes
print(f"\nCargando en lotes de {BATCH_SIZE}...")
cargar_documentos(productos + clientes + pedidos)

# 4. Verificación
info = SESSION.get(DB_URL, timeout=10).json()
esperado = len(productos) + len(clientes) + len(pedidos)
print(f"\nDocumentos en la BD: {info['doc_count']} (esperados de esta simulación: {esperado})")

# 5. Documentos reales (evidencia para la demostración)
import json
print("\n--- Ejemplos de documentos guardados (acceso directo por _id) ---")
for doc_id in ("PROD-000001", "CLI-000001", "PED-000001"):
    doc = SESSION.get(f"{DB_URL}/{doc_id}", timeout=10).json()
    print(f"\n{doc_id}:\n{json.dumps(doc, indent=2, ensure_ascii=False)}")

print("\n=== CARGA COMPLETA ===")